In [ ]:
!pip install openai nest-asyncio llama-index

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.4/253.4 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.5 MB/s eta 0:00:00


# Installing required Packages

In [ ]:
import nest_asyncio
nest_asyncio.apply()

# Importing important libraries

In [ ]:
import os
from llama_index.core import (
    Settings,
    SimpleDirectoryReader,
    ServiceContext,
    VectorStoreIndex,
    get_response_synthesizer)
from llama_index.llms.openai import OpenAI
from llama_index.core.evaluation import (
    DatasetGenerator,
    FaithfulnessEvaluator,
    RelevancyEvaluator)
from llama_index.core.postprocessor import SimilarityPostprocessor
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Downloaing a text file and saving it

In [ ]:
# download Avenger's script which is used through out the notebook
import requests
response = requests.get("https://www.dailyscript.com/scripts/Avengers.html")
essay_txt = response.text
with open("Avengers_script.txt", "w") as f:
  f.write(essay_txt)

# Generating questions

In [ ]:
# build service context
Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0.1)

# build documents
documents = SimpleDirectoryReader(input_files=['Avengers_script.txt']).load_data()

# define generator, generate questions
data_generator = DatasetGenerator.from_documents(documents)

questions = data_generator.generate_questions_from_nodes()

/usr/local/lib/python3.11/dist-packages/llama_index/core/evaluation/dataset_generation.py:200: DeprecationWarning: Call to deprecated class DatasetGenerator. (Deprecated in favor of `RagDatasetGenerator` which should be used instead.)
  return cls(
/usr/local/lib/python3.11/dist-packages/llama_index/core/evaluation/dataset_generation.py:296: DeprecationWarning: Call to deprecated class QueryResponseDataset. (Deprecated in favor of `LabelledRagDataset` which should be used instead.)
  return QueryResponseDataset(queries=queries, responses=responses_dict)


In [ ]:
questions

['Who is the author of "The Avengers" screenplay as mentioned in the document?',
 'Describe the setting of the opening scene in the script. What elements are used to create the atmosphere?',
 'What is the significance of the red phone box in the context of the script?',
 'Identify the character introduced in the scene and describe her appearance and demeanor.',
 'What is the first line spoken by Emma Peel in the script, and what does it suggest about her character?',
 "How does the official voice respond to Emma's initial dialogue, and what does this indicate about the nature of her mission?",
 'What action does Emma take after hanging up the phone, and what does the notice she hangs outside the phone box say?',
 'Describe the transition that occurs when Emma enters the phone box. What happens to the setting?',
 'What details are provided about the laboratory where Emma descends? How is it described?',
 'Based on the context provided, what themes or motifs can be inferred from the open

# Generating Response

In [ ]:
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine


# build index
index = VectorStoreIndex.from_documents(documents)

# configure retriever
retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=10,
)

# configure response synthesizer
response_synthesizer = get_response_synthesizer()

# assemble query engine
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer,
    node_postprocessors=[SimilarityPostprocessor(similarity_cutoff=0.7)],
)

# query
response = query_engine.query(questions[2])
print(response)

The red phone box serves as a pivotal element in the script, acting as a trigger for potential explosions. Its ringing prompts suspicion and urgency, particularly from Steed, who warns Emma not to answer it. This moment highlights the tension and danger present in the scene, as the phone's call is linked to a larger threat. Additionally, the phone box symbolizes a connection to the outside world and the unfolding mystery, ultimately leading to a significant revelation when Steed answers the call and learns about Dr. Darling's disappearance.


# Evaluation:
# Response + Source Nodes (Context)
# Query + Response + Source Nodes (Context)
# Query + Response + Individual Source Nodes (Context)

In [ ]:
# build index
index = VectorStoreIndex.from_documents(documents)

# define evaluator
evaluator = FaithfulnessEvaluator()

# query index
query_engine = index.as_query_engine()
response = query_engine.query(questions[2])
eval_result = evaluator.evaluate_response(response=response)
print(str(eval_result.passing))

True


In [ ]:
# define evaluator
evaluator = RelevancyEvaluator()

# query index
query_engine = index.as_query_engine()
query = questions[2]
response = query_engine.query(query)
eval_result = evaluator.evaluate_response(query=query, response=response)
print(str(eval_result))

query='What is the significance of the red phone box in the context of the script?' contexts=['Choirboys walk out from the church.  Nearby in the deserted\r\n\tvillage street.  A red PHONE BOX.  Which ...\r\n\r\n\tRING-RING ... Starts to RING.\r\n\r\n\t\t\t\t\tEMMA\r\n\t\t\tWho could that be?\r\n\r\n\tA ROLL of THUNDER.  Steed looks up: a clear sky.  He\'s puzzled.  Suddenly\r\n\tsuspicious.  As Emma moves to the phone.\r\n\r\n\t\t\t\t\tSTEED\r\n\t\t\tNo -- don\'t answer it ...\r\n\r\n\tHe pulls her back.  Emma looks at him.\r\n\r\n\t\t\t\t\tSTEED\r\n\t\t\tThat\'s it. The phones trigger\r\n\t\t\tthe explosions --\r\n\r\n\tRING-RING ... Another ROLL of THUNDER.  Steed connects the two as -- an\r\n\tangelic CHOIRBOY walks towards the phone ...\r\n\r\n\tRING-RING ... A LOUDER ROLL of THUNDER.  As the Choirboy nears the\r\n\tPHONE, Steed shouts --\r\n\r\n\t\t\t\t\tSTEED\r\n\t\tDon\'t -- don\'t answer it -- !\r\n\r\n\r\n190     CLOSEUP - PHONE\r\n\r\n\tRING-RING -- the PHONE in the f.g. as 

In [ ]:
# define evaluator
evaluator = RelevancyEvaluator()

# query index
query_engine = index.as_query_engine()
query = questions[2]
response = query_engine.query(query)
response_str = response.response
for source_node in response.source_nodes:
    eval_result = evaluator.evaluate(
        query=query,
        response=response_str,
        contexts=[source_node.get_content()],
    )
    print(str(eval_result.passing))

True
False
